# 03 — Official business activity (test)

The Ministry of Commerce publishes the **registered activities of every commercial registration**
("ارتباط السجلات التجارية القائمة مع التصنيف الوطني للأنشطة الاقتصادية 2025", open.data.gov.sa).
Each row is one activity of one registration: national ID, section (level 1), ISIC class (level 2, 4 digits)
and SSIC activity (level 3, 6 digits). A business usually has several activities.

**Question of this notebook:** can these official codes replace (or check) the SSIC codes that
`01e_ssic_classification` guesses from the business name?

1. **Coverage:** how many of our businesses are in the file?
2. **Leakage check:** if the file lists only businesses still active in 2025, businesses that closed are
   missing from it, so "found in the file" would mean "survived". Coverage must be about the same for
   survived and disappeared businesses.
3. **Main activity:** the file has no "main activity" column, so two rules are compared: the section most of
   the activities belong to (`main_*`) and the first activity listed (`first_*`).
4. **Accuracy of the name classifier**, measured against the official codes.

If it works, the cells move into `01e_ssic_classification`.

**Needs:** `ssic_by_business.csv` from `01e_ssic_classification` Cell 19 (run 01b to 01e first on the **same Colab server**,
or upload the file to `DATA_DIR`), and the activity file in the Drive folder Darb DataSets.
**Rules:** raw files never changed; results saved in the data folder, never in Git; clear outputs before committing.

### Cell 1 — Setup

In [ ]:
import os
import gc
import re
from pathlib import Path

import numpy as np
import pandas as pd

IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    %pip install -q gdown python-calamine
    DATA_DIR = Path('/content/darb-data')
else:
    DATA_DIR = Path(os.environ.get('DARB_DATA', str(Path.home() / 'darb-data')))

print('Running on Colab' if IN_COLAB else 'Running locally', '| data folder:', DATA_DIR)

### Cell 2 — The files
- **Activity file:** `CR_ISIC_2025.csv` (4.2 GB). A zipped copy (`CR_ISIC_2025.zip`) works too and uploads
  ~10× faster: change `ISIC_FILE` to the zip name. **Fill in `ISIC_DRIVE_ID`** with its Google Drive ID
  (the long code in its share link).
- **SSIC file:** the official list, to check the codes.
- **`ssic_by_business.csv`:** from `01e_ssic_classification` (one row per business: cohort, label, name-based SSIC code).

In [ ]:
ISIC_FILE = 'CR_ISIC_2025.zip'      # the zipped CSV in Drive (pandas reads it directly)
ISIC_DRIVE_ID = '1b5WUOsiKE6o6YNZoNo8tTs92DI7-8hnn'   # Drive ID of the file in Darb DataSets
ISIC_PATH = DATA_DIR / 'MC_ISIC' / ISIC_FILE

SSIC_FILE = 'National Classification for the Economic Activities(SSIC)_15 Sep_2026_AR.xlsx'
SSIC_PATH = DATA_DIR / 'Gastat' / SSIC_FILE
LABELS_PATH = DATA_DIR / 'ssic_by_business.csv'

if IN_COLAB:
    import gdown
    for path, drive_id in [(ISIC_PATH, ISIC_DRIVE_ID), (SSIC_PATH, '1wlS-84Ojsiv0UP8sjmZmNcENU_M8J0wZ')]:
        if not path.exists() and drive_id:
            path.parent.mkdir(parents=True, exist_ok=True)
            print('downloading', path.name, '(the activity file takes several minutes)')
            gdown.download(id=drive_id, output=str(path), quiet=True)

for path in [ISIC_PATH, SSIC_PATH, LABELS_PATH]:
    print(f'{path.name:<75}', f'found ({path.stat().st_size / 1e6:,.0f} MB)' if path.exists() else 'MISSING')

if not ISIC_PATH.exists():
    raise FileNotFoundError('Activity file missing: upload it to Darb DataSets and fill in ISIC_DRIVE_ID.')
if not LABELS_PATH.exists():
    raise FileNotFoundError('ssic_by_business.csv missing: run 01b_all_quarters to 01e_ssic_classification on the same '
                            'Colab server first, or upload the file to ' + str(DATA_DIR))

### Cell 3 — Read the activity file (in pieces, only the codes)
The file is too big to read at once, so it is read 2 million rows at a time, keeping only the ID and the three
codes as numbers (small in memory). Each row keeps its position (`order`), so "the first activity listed" is known.
Rows with a missing or non-numeric code are counted, not guessed.

In [ ]:
NAMES = ['id', 'cr_number', 'section_code', 'section_desc', 'class_code', 'class_desc', 'activity_code', 'activity_desc']
CODES = ['id', 'section_code', 'class_code', 'activity_code']

parts, section_descs, rows, bad = [], set(), 0, 0
reader = pd.read_csv(ISIC_PATH, names=NAMES, header=0, usecols=CODES + ['section_desc'],
                     dtype=str, encoding='utf-8-sig', chunksize=2_000_000)
for chunk in reader:
    section_descs |= set(chunk[['section_code', 'section_desc']].drop_duplicates().itertuples(index=False, name=None))
    part = chunk[CODES].apply(lambda column: pd.to_numeric(column.str.strip(), errors='coerce'))
    ok = part.notna().all(axis=1)
    bad += int((~ok).sum())
    part = part[ok].astype({'id': 'int64', 'section_code': 'int8', 'class_code': 'int16', 'activity_code': 'int32'})
    part['order'] = part.index.to_numpy(dtype='int64')
    parts.append(part)
    rows += len(chunk)
    print(f'  read {rows:>12,} rows')

act = pd.concat(parts, ignore_index=True)
del parts, reader
gc.collect()
act['division'] = (act['class_code'] // 100).astype('int8')     # 4-digit class -> 2-digit division (0111 -> 01)

print()
print(f'Activity rows: {rows:,} | rows with a missing or wrong code (left out): {bad:,}')
print(f'Businesses (IDs): {act["id"].nunique():,} | memory: {act.memory_usage(deep=True).sum() / 1e9:.2f} GB')

### Cell 4 — Do the codes match the official SSIC?
- **Sections:** the file numbers them 1–21. SSIC uses Arabic letters, which `01e_ssic_classification` maps to A–U in
  their order. The table shows each number with the file's own name and the SSIC name of the letter: they must
  describe the same section.
- **Divisions:** every (section, division) pair must exist in SSIC.

In [ ]:
LETTERS = np.array(list('ABCDEFGHIJKLMNOPQRSTU'))

# The official SSIC list (same reading as 01e_ssic_classification Cell 16)
raw = pd.read_excel(SSIC_PATH, engine='calamine', header=None, dtype=str)
header_row = raw.index[raw[0].fillna('').str.strip() == 'رمز الباب'][0]
ssic = raw.iloc[header_row + 1:, :4].copy()
ssic.columns = ['section_code', 'section_name', 'division_code', 'division_name']
ssic = ssic.dropna(subset=['division_code']).apply(lambda column: column.str.strip())
arabic_sections = list(dict.fromkeys(ssic['section_code']))
ssic['letter'] = ssic['section_code'].map(dict(zip(arabic_sections, LETTERS)))
section_name = ssic.drop_duplicates('letter').set_index('letter')['section_name']
official_pairs = set(zip(ssic['letter'], ssic['division_code'].astype(int)))
del raw

def squash(text):
    return re.sub(r'\s+', ' ', str(text)).strip()

codes = pd.DataFrame(sorted(section_descs, key=lambda p: int(p[0])), columns=['code', 'name in the file'])
codes['letter'] = LETTERS[codes['code'].astype(int) - 1]
codes['SSIC name of the letter'] = codes['letter'].map(section_name)
codes['same name'] = [squash(a) == squash(b) for a, b in zip(codes['name in the file'], codes['SSIC name of the letter'])]
with pd.option_context('display.width', 250, 'display.max_colwidth', 70):
    print(codes.to_string(index=False))

# (section, division) as one number: section number x 100 + division, e.g. C14 -> 314
official_keys = [(int(np.where(LETTERS == letter)[0][0]) + 1) * 100 + division for letter, division in official_pairs]
pair_ok = (act['section_code'].astype('int32') * 100 + act['division']).isin(official_keys).to_numpy()
print()
print(f'Activity rows whose (section, division) is in SSIC: {pair_ok.mean():.2%}')
if not pair_ok.all():
    wrong = act.loc[~pair_ok, ['section_code', 'division']].value_counts().head(10)
    print('Most common pairs not in SSIC (section number, division):')
    print(wrong.to_string())
del pair_ok

### Cell 5 — One row per business
For each ID:
- `n_activities`, `n_sections`, `sections_all` (e.g. `C|G`)
- **`main_section`**: the section most of its activities belong to (a tie goes to the one listed first);
  `main_division`: the most common division inside that section
- **`first_section`, `first_division`, `first_activity`**: the first activity listed in the file

In [ ]:
act = act.sort_values('order', kind='stable').reset_index(drop=True)
by_id = act.groupby('id', sort=False)
business = pd.DataFrame({'n_activities': by_id.size()})

first = act.drop_duplicates('id').set_index('id')
business['first_section'] = LETTERS[first['section_code'].reindex(business.index).to_numpy() - 1]
business['first_division'] = first['division'].reindex(business.index).astype(int).astype(str).str.zfill(2)
business['first_activity'] = first['activity_code'].reindex(business.index).astype(int).astype(str).str.zfill(6)

# Main section: most activities, tie -> listed first
per_section = (act.groupby(['id', 'section_code'], sort=False)
               .agg(n=('order', 'size'), first=('order', 'min')).reset_index())
business['n_sections'] = per_section.groupby('id').size().reindex(business.index)
bits = per_section.assign(bit=np.left_shift(1, per_section['section_code'].astype('int64') - 1)).groupby('id')['bit'].sum()
mask_text = {m: '|'.join(LETTERS[[i for i in range(21) if m >> i & 1]]) for m in bits.unique()}
business['sections_all'] = bits.reindex(business.index).map(mask_text)
main = (per_section.sort_values(['id', 'n', 'first'], ascending=[True, False, True])
        .drop_duplicates('id').set_index('id')['section_code'])
business['main_section'] = LETTERS[main.reindex(business.index).to_numpy() - 1]

# Main division: most common division inside the main section, tie -> listed first
per_division = (act.groupby(['id', 'section_code', 'division'], sort=False)
                .agg(n=('order', 'size'), first=('order', 'min')).reset_index())
per_division = per_division[per_division['section_code'].to_numpy() == main.reindex(per_division['id']).to_numpy()]
main_division = (per_division.sort_values(['id', 'n', 'first'], ascending=[True, False, True])
                 .drop_duplicates('id').set_index('id')['division'])
business['main_division'] = main_division.reindex(business.index).astype(int).astype(str).str.zfill(2)
del by_id, first, per_section, per_division, bits, main, main_division
gc.collect()

print(f'Businesses: {len(business):,}')
print()
print('Activities per business:')
print(business['n_activities'].describe(percentiles=[.25, .5, .75, .9, .99]).round(1).to_string())
print()
print(f'Businesses with activities in more than one section: {(business["n_sections"] > 1).mean():.1%}')
print(f'Main section = first section: {(business["main_section"] == business["first_section"]).mean():.1%}')
print()
print('Main section of the businesses:')
print(business['main_section'].map(lambda l: f'{l} {section_name.get(l, "")}').value_counts().to_string())

### Cell 6 — Coverage, and the leakage check
Joins the businesses of `01e_ssic_classification` (`entity_id`) to the file's IDs.

**Leakage check:** in the labelled cohorts, the share found in the file should be about the same for
**survived** and **disappeared** businesses. If disappeared businesses are found much less often, the file lists
only businesses still active at its date: then "found in the file" leaks the label, and the official code can
only be used where it does not depend on survival (for example as an answer key to improve the name
classifier), not directly as a model feature.

The coverage by start quarter also hints at the file's date: businesses that started after it are not in it.

In [ ]:
labels = pd.read_csv(LABELS_PATH, dtype=str, encoding='utf-8-sig', keep_default_na=False)
labels['id_number'] = pd.to_numeric(labels['entity_id'], errors='coerce').fillna(-1).astype('int64')
labels['found'] = labels['id_number'].isin(business.index)
labelled = labels[labels['cohort_quality'] != 'cannot_judge']
OUTCOME = {'1': 'survived', '0': 'disappeared', '': 'undecided'}

print(f'All businesses: {labels["found"].mean():.1%} found in the activity file ({labels["found"].sum():,} of {len(labels):,})')
print(f'Labelled cohorts: {labelled["found"].mean():.1%} found ({labelled["found"].sum():,} of {len(labelled):,})')
print()
print('Coverage per labelled cohort:')
print(labelled.groupby(['cohort', 'cohort_quality'])['found'].agg(businesses='size', found='mean')
      .assign(found=lambda t: t['found'].map('{:.1%}'.format)).to_string())
print()
by_outcome = labelled.groupby(labelled['survived_1y'].map(OUTCOME))['found'].agg(businesses='size', found='mean')
print('Coverage by outcome (the leakage check):')
print(by_outcome.assign(found=lambda t: t['found'].map('{:.1%}'.format)).to_string())
gap = by_outcome.loc['survived', 'found'] - by_outcome.loc['disappeared', 'found']
print()
print(f'Gap survived - disappeared: {gap:+.1%}')
print('⚠ Large gap: the file likely lists only businesses active at its date -> "found" leaks the label.'
      if gap > 0.10 else '✓ Small gap: being in the file does not depend much on survival.')
print('(rule of thumb: a gap above 10 points is a problem)')
print()
recent = labels[(labels['cohort'] >= '2023Q1') & (labels['cohort'] <= '2026Q4')]
print('Coverage by start quarter (hints at the date of the file):')
print(recent.groupby('cohort')['found'].agg(businesses='size', found='mean')
      .assign(found=lambda t: t['found'].map('{:.1%}'.format)).to_string())

### Cell 7 — How accurate is the name classifier?
For labelled businesses found in the file, the name-based code of the **latest name** (`ssic_latest_*` from
`01e_ssic_classification`, the name closest to the file's date) is compared with the official one, per `ssic_flag`:
- **= main / = first:** same section as the main / first official activity
- **in any:** the name's section is one of the business's official sections
- **division (ok only):** same division as the main official division

It also shows how many businesses the name classifier could not classify get an official code.

In [ ]:
both = labelled[labelled['found']].join(business, on='id_number')
has_section = both[both['ssic_latest_section'] != '']
in_any = [s in a.split('|') for s, a in zip(has_section['ssic_latest_section'], has_section['sections_all'])]
section_check = (has_section.assign(**{
                     '= main': has_section['ssic_latest_section'] == has_section['main_section'],
                     '= first': has_section['ssic_latest_section'] == has_section['first_section'],
                     'in any': in_any})
                 .groupby('ssic_latest_flag')[['= main', '= first', 'in any']].mean())
section_check.insert(0, 'businesses', has_section['ssic_latest_flag'].value_counts())
print('Section agreement, name classifier vs official (labelled businesses found in the file):')
print(section_check.to_string(formatters={c: '{:.1%}'.format for c in ['= main', '= first', 'in any']}))

ok_div = both[(both['ssic_latest_flag'] == 'ok') & (both['ssic_latest_division'] != '')]
print()
print(f'Division agreement (flag ok, {len(ok_div):,} businesses): '
      f'= main {(ok_div["ssic_division"].str.zfill(2) == ok_div["main_division"]).mean():.1%} | '
      f'= first {(ok_div["ssic_division"].str.zfill(2) == ok_div["first_division"]).mean():.1%}')
print()
no_code = labelled[labelled['ssic_latest_flag'].isin(['unclassified', 'ambiguous', 'names_disagree'])]
print(f'Labelled businesses without a sure name-based section: {len(no_code):,} | '
      f'found in the activity file: {no_code["found"].mean():.1%}')

### Cell 8 — Save
Saves `official_activity_by_business.csv` (one row per ID in the activity file) in the data folder, never in Git.

In [ ]:
OUT = DATA_DIR / 'official_activity_by_business.csv'
business.to_csv(OUT, index_label='id', encoding='utf-8-sig')
print(f'Saved {len(business):,} businesses to {OUT}')

## Part 2 — Better keywords from the official activities (no second model)

Part 1 showed that the official sector cannot be a model feature (leakage). So every business keeps getting its
sector **from its name**, with keyword rules, exactly like `isic_classifier.py`. This part only makes the keyword
lists **bigger and checked**:

1. **Keywords from real names (automatic):** for every word in the business names, count how often its businesses
   are really in each sector (the official file is the answer key). A word that points to one sector most of the
   time (e.g. 80%+) and appears often enough becomes a keyword for that sector, e.g. "موضه" → the sector most
   "موضه" businesses really have.
2. **Keywords from the SSIC activity names (automatic):** a word used in the activity names of only **one**
   section (e.g. "خياطه") becomes a keyword for it. Used only when step 1 finds nothing.
3. **Honest test:** the keywords are learned on 80% of the businesses and tested on the other 20%, which were never
   used to learn them. Accuracy is compared with the current rules, and checked separately for survived and
   disappeared businesses (it must work the same for both).
4. **Rules to review:** the current rules that most often disagree with the official sector.

**Needs:** Part 1 run first, `business_names.csv` from `01e_ssic_classification` Cell 19 (the latest name of each
business; private, data folder only), and `isic_classifier.py` in `/content` (same as `01e_ssic_classification` Cell 17).

### Cell 9 — Names, and the learn / test split
Each business found in the activity file gets its name and its official sector. A fixed 20% of them (chosen from
the ID, so the same every run) is kept aside as the **test set**; keywords are learned only from the other 80%.

**Which name:** keywords are learned and tested on the **latest** name, because the official file is from the
end of 2025 and the latest name belongs with it. The final codes for the model (Cell 14) are given to the **start**
name, what is known on day one.

In [ ]:
import sys
SRC_DIR = Path('/content') if IN_COLAB else Path.cwd().parent / 'src'
if not (SRC_DIR / 'isic_classifier.py').exists():
    raise FileNotFoundError(f'Put isic_classifier.py in {SRC_DIR} first (the %%writefile cell, as in 01e_ssic_classification).')
sys.path.insert(0, str(SRC_DIR))
from isic_classifier import normalize

NAMES_PATH = DATA_DIR / 'business_names.csv'
if not NAMES_PATH.exists():
    raise FileNotFoundError('business_names.csv missing: run 01e_ssic_classification Cell 19 again on this Colab server.')

names = pd.read_csv(NAMES_PATH, dtype=str, encoding='utf-8-sig', keep_default_na=False)
if 'start_name' not in names.columns:
    raise ValueError('business_names.csv is from an older 01e_ssic_classification: run its Cell 19 again.')
data = labels.merge(names, on='entity_id', how='left')
data[['start_name', 'latest_name']] = data[['start_name', 'latest_name']].fillna('')
# Learning and testing use the LATEST name: the closest to the official file's date (end of 2025).
# The final codes for the model (Cell 14) use the START name.
data['name'] = data['latest_name']
data['found'] = data['id_number'].isin(business.index)
data = data.join(business[['main_section', 'main_division', 'sections_all']], on='id_number')
data['test'] = pd.util.hash_pandas_object(data['entity_id'], index=False).to_numpy() % 5 == 0

known = data[data['found'] & (data['name'] != '')]
learn, test = known[~known['test']], known[known['test']]
print(f'Businesses: {len(data):,} | with a name and an official sector: {len(known):,}')
print(f'Learn: {len(learn):,} | test: {len(test):,} (never used to learn)')

### Cell 10 — Keywords from real names
Each name is cleaned like the classifier does (`normalize`), the prefixes ال / لل / وال / بال / فال are removed,
and legal-form words (مؤسسه, شركه, ذات مسؤوليه محدوده …) are dropped because every sector has them. Single words
and two-word phrases are counted.

A word becomes a keyword when:
- it appears in at least `MIN_SUPPORT` names (so it is a real pattern, not one business), and
- at least `MIN_PRECISION` of its businesses are in the same section.

Its division is kept when at least `MIN_DIVISION_PRECISION` of its businesses are in the same division; otherwise
the keyword gives the section only. Personal names (محمد, عبدالله …) are removed by the rule itself: they appear
in every sector, so no sector reaches 80%.

In [ ]:
MIN_SUPPORT = 30
MIN_PRECISION = 0.80
MIN_DIVISION_PRECISION = 0.70

STRIP = ('وال', 'بال', 'فال', 'لل', 'ال')
STOP = {normalize(w) for w in [
    'مؤسسة', 'مؤسسه', 'شركة', 'شركه', 'ذات', 'مسؤولية', 'مسئولية', 'محدودة', 'شخص', 'واحد', 'مساهمة', 'مبسطة',
    'مختلطة', 'فرع', 'تضامنية', 'توصية', 'بسيطة', 'قابضة', 'مهنية', 'فردية', 'اجنبية', 'خليجية', 'وشركاه',
    'وشريكه', 'واولاده', 'اولاده', 'وابناءه', 'وابنائه', 'واخوانه', 'تجارة', 'تجاري', 'تجارية', 'خدمات', 'عامة',
    'مجموعة', 'سعودية', 'السعودية']}

def strip_prefix(word):
    for prefix in STRIP:
        if word.startswith(prefix) and len(word) - len(prefix) >= 3:
            return word[len(prefix):]
    return word

def words_of(name):
    """Words and two-word phrases of a name, cleaned like the classifier, without legal-form words."""
    tokens = [strip_prefix(w) for w in normalize(name).split()]
    tokens = [w for w in tokens if len(w) >= 3 and w not in STOP]
    return list(dict.fromkeys(tokens + [f'{a} {b}' for a, b in zip(tokens, tokens[1:])]))

def keyword_table(rows):
    """rows: one row per (business, word) with its official main_section and main_division."""
    total = rows.groupby('word').size().rename('n')
    by_section = rows.groupby(['word', 'main_section']).size().rename('n_section').reset_index()
    best = by_section.sort_values('n_section', ascending=False).drop_duplicates('word').set_index('word')
    table = best.join(total)
    table['precision'] = table['n_section'] / table['n']
    table = table[(table['n'] >= MIN_SUPPORT) & (table['precision'] >= MIN_PRECISION)]
    by_division = (rows[rows['word'].isin(table.index)]
                   .groupby(['word', 'main_section', 'main_division']).size().rename('n_division').reset_index())
    by_division = by_division[by_division['main_section'].to_numpy() == table['main_section'].reindex(by_division['word']).to_numpy()]
    best_division = by_division.sort_values('n_division', ascending=False).drop_duplicates('word').set_index('word')
    table = table.join(best_division[['main_division', 'n_division']])
    table['division_precision'] = table['n_division'] / table['n']
    table['division'] = np.where(table['division_precision'] >= MIN_DIVISION_PRECISION, table['main_division'], '')
    table = (table.rename(columns={'main_section': 'section'})
             [['section', 'division', 'n', 'precision', 'division_precision']]
             .assign(source='names').sort_values('n', ascending=False))
    return table, total

learn_rows = (learn[['main_section', 'main_division']]
              .assign(word=learn['name'].map(words_of)).explode('word').dropna(subset=['word']))
name_keywords, word_count = keyword_table(learn_rows)
del learn_rows
gc.collect()

print(f'Keywords from names: {len(name_keywords):,} '
      f'(single words: {(~name_keywords.index.str.contains(" ")).sum():,}, '
      f'phrases: {name_keywords.index.str.contains(" ").sum():,}) | with a division: {(name_keywords["division"] != "").mean():.0%}')
print()
print('Keywords per section:')
print(name_keywords['section'].map(lambda l: f'{l} {section_name.get(l, "")}').value_counts().to_string())
print()
print('The 10 most used keywords of each section (for a quick look):')
for letter, group in name_keywords.groupby('section'):
    print(f'{letter}: ' + ', '.join(f'{w} ({p:.0%})' for w, p in group['precision'].head(10).items()))

### Cell 10b — Keyword review file (nothing is removed)
Some keywords may be **family or person names** (e.g. a family that registered many businesses in one sector),
not activities. Nothing is removed here: every keyword stays in use. This cell only writes a review file with an
empty **`decision`** column for a person to fill in later (`keep` or `drop`).

To help the review, `note` marks two kinds of phrases (only a hint, not a decision):
- `no activity word`: neither of its two words is a keyword by itself (e.g. two personal names)
- `repeats a keyword`: one of its words is already a keyword for the same section (e.g. "… مقاولات"), so the
  phrase adds nothing

Saves `ssic_keywords_review.xlsx` in the data folder (never in Git: it can contain names) with the sheets
`single_words` and `phrases`, and shows a download link.

In [ ]:
review = name_keywords.reset_index().rename(columns={'index': 'word'})
review['type'] = np.where(review['word'].str.contains(' '), 'phrase', 'single word')
review['section name'] = review['section'].map(section_name)
single_section = review.loc[review['type'] == 'single word'].set_index('word')['section']

def note_of(row):
    if row['type'] != 'phrase':
        return ''
    sections = [single_section.get(w) for w in row['word'].split()]
    if all(s is None for s in sections):
        return 'no activity word'
    if row['section'] in sections:
        return 'repeats a keyword'
    return ''

review['note'] = review.apply(note_of, axis=1)
review['decision'] = ''                       # to fill in later: keep / drop
columns = ['word', 'section', 'section name', 'division', 'n', 'precision', 'division_precision', 'note', 'decision']

REVIEW_PATH = DATA_DIR / 'ssic_keywords_review.xlsx'
with pd.ExcelWriter(REVIEW_PATH) as writer:
    review[review['type'] == 'single word'][columns].to_excel(writer, sheet_name='single_words', index=False)
    review[review['type'] == 'phrase'][columns].to_excel(writer, sheet_name='phrases', index=False)

print(f'Single words: {(review["type"] == "single word").sum():,} | phrases: {(review["type"] == "phrase").sum():,}')
print('Phrase notes (hints only, nothing removed):')
print(review.loc[review['type'] == 'phrase', 'note'].replace('', '(none)').value_counts().to_string())

import base64
from IPython.display import HTML, display
XLSX = 'application/vnd.openxmlformats-officedocument.spreadsheetml.sheet'
data64 = base64.b64encode(REVIEW_PATH.read_bytes()).decode()
display(HTML(f'<a download="{REVIEW_PATH.name}" href="data:{XLSX};base64,{data64}">⬇ Download {REVIEW_PATH.name}</a>'))

### Cell 10c — Apply the keyword decisions (keep / drop)
Reads the review file after a person filled in the `decision` column (`ssic_keywords_decisions_reviewed.xlsx`,
sheets `single_words` and `phrases`) and **stops using every keyword marked `Drop`** (mostly family and person
names). Only the columns `word` and `decision` are read.

- Keywords marked `Keep`, or with no decision, stay in use.
- A keyword learned now but **not in the file** (e.g. after the data changed) stays in use; its count is printed.
- `USE_DECISIONS = False` turns this off (all keywords used, as before).

**The file:** upload it to Darb DataSets and fill in `DECISIONS_DRIVE_ID`, or upload it by hand to the data folder.
It contains names: data folder only, **never in Git**.

In [ ]:
USE_DECISIONS = True
DECISIONS_FILE = 'ssic_keywords_decisions_reviewed.xlsx'
DECISIONS_DRIVE_ID = ''                       # <- fill in: Drive ID of the file (or upload it to DATA_DIR)
DECISIONS_PATH = DATA_DIR / DECISIONS_FILE

if USE_DECISIONS:
    if IN_COLAB and not DECISIONS_PATH.exists() and DECISIONS_DRIVE_ID:
        import gdown
        gdown.download(id=DECISIONS_DRIVE_ID, output=str(DECISIONS_PATH), quiet=True)
    if not DECISIONS_PATH.exists():
        raise FileNotFoundError(f'{DECISIONS_FILE} missing: fill in DECISIONS_DRIVE_ID or upload it to {DATA_DIR} '
                                '(or set USE_DECISIONS = False to use all keywords).')
    sheets = pd.read_excel(DECISIONS_PATH, sheet_name=None, engine='calamine', dtype=str)
    decisions = pd.concat([sheet[['word', 'decision']] for sheet in sheets.values()], ignore_index=True)
    decisions['word'] = decisions['word'].fillna('').str.strip()
    decisions['decision'] = decisions['decision'].fillna('').str.strip().str.lower()
    dropped = set(decisions.loc[decisions['decision'] == 'drop', 'word'])

    is_phrase = name_keywords.index.str.contains(' ')
    removed = name_keywords.index.isin(dropped)
    not_in_file = (~name_keywords.index.isin(decisions['word'])).sum()
    print(f'Decisions file: {len(decisions):,} keywords | marked Drop: {len(dropped):,}')
    print(f'Removed now: {removed.sum():,} (single words: {(removed & ~is_phrase).sum():,}, '
          f'phrases: {(removed & is_phrase).sum():,})')
    print(f'Learned now but not in the file (kept): {not_in_file:,}')
    name_keywords = name_keywords[~removed]
    print(f'Keywords in use: {len(name_keywords):,}')
else:
    print('USE_DECISIONS = False: all keywords are used.')

### Cell 11 — Keywords from the SSIC activity names
The official names of the 2,847 SSIC activities (and of their classes) are split into words the same way. A word
used in **one section only** becomes a keyword for that section (and for its division if it is in one division
only). These are formal words ("خياطه", "تشطيب"), so they are used only when no keyword from real names is found,
and only for words that are **rare** in the names: if a word is common in names but its businesses are spread over
several sectors, the names have already shown it is ambiguous. Their accuracy is shown separately in Cell 12.

In [ ]:
raw = pd.read_excel(SSIC_PATH, engine='calamine', header=None, dtype=str)
header_row = raw.index[raw[0].fillna('').str.strip() == 'رمز الباب'][0]
full = raw.iloc[header_row + 1:].copy()
full.columns = ['section_code', 'section_name', 'division_code', 'division_name', 'group_code', 'group_name',
                'class_code', 'class_name', 'activity_code', 'activity_name', 'activity_description',
                'supervising_authority', 'licensing_authority']
full = full.dropna(subset=['activity_code']).apply(lambda column: column.str.strip())
full['letter'] = full['section_code'].map(dict(zip(arabic_sections, LETTERS)))
full['division'] = full['division_code'].str.zfill(2)
del raw

GENERIC_SSIC = {normalize(w) for w in [
    'انشطة', 'نشاط', 'صنع', 'بيع', 'تجزئة', 'جملة', 'خدمات', 'اخرى', 'الاخرى', 'غير', 'مصنفة', 'موضع', 'اخر',
    'يشمل', 'تشمل', 'مثل', 'وغيرها', 'انواع', 'جميع', 'مختلف', 'باستثناء', 'المتخصصة', 'متخصصة', 'المتاجر',
    'متاجر', 'محلات', 'محل', 'منتجات', 'المنتجات', 'مواد', 'المواد', 'تشغيل', 'اصلاح', 'صيانة', 'تركيب',
    'تاجير', 'استئجار', 'الاسواق', 'اسواق', 'الاكشاك', 'تقديم', 'الخاصة', 'خاصة', 'العامة']}

ssic_rows = (full[['letter', 'division']]
             .assign(word=(full['activity_name'] + ' ' + full['class_name']).map(lambda t: [
                 w for w in words_of(t) if ' ' not in w and w not in GENERIC_SSIC]))
             .explode('word').dropna(subset=['word']).drop_duplicates())
sections_of_word = ssic_rows.groupby('word')['letter'].nunique()
divisions_of_word = ssic_rows.groupby('word')['division'].nunique()
one_section = sections_of_word.index[sections_of_word == 1]
description_keywords = (ssic_rows[ssic_rows['word'].isin(one_section)].drop_duplicates('word').set_index('word')
                        .rename(columns={'letter': 'section'}))
description_keywords['division'] = np.where(divisions_of_word.reindex(description_keywords.index) == 1,
                                            description_keywords['division'], '')
description_keywords = description_keywords.assign(source='description')
# Not used when the real names already show the word is ambiguous (common in names, but no sector reaches 80%)
common_in_names = word_count.index[word_count >= MIN_SUPPORT]
description_keywords = description_keywords[~description_keywords.index.isin(common_in_names)]
# Switch: True = use these keywords, False = do not use them (Cell 12 shows they are right only ~22% of the time)
USE_SSIC_NAME_KEYWORDS = False
if not USE_SSIC_NAME_KEYWORDS:
    description_keywords = description_keywords.iloc[0:0]

print(f'Keywords from SSIC activity names (only for words that are rare in names): {len(description_keywords):,}')
print('Switch USE_SSIC_NAME_KEYWORDS =', USE_SSIC_NAME_KEYWORDS)
print('Examples:', ', '.join(f'{w}→{s}{d}' for w, s, d in
                              description_keywords[['section', 'division']].head(15).itertuples(name=None)))

### Cell 12 — The honest test: rules vs keywords
On the **test set** (20%, never used to learn), each method gives a section or nothing:

| Method | What it does |
|---|---|
| `rules` | the current rules (`ssic_flag` ok or division_unclear) |
| `keywords` | the new keywords (from names, else from SSIC names) |
| `rules, then keywords` | the rules; keywords only where the rules have no sure section |
| `keywords, then rules` | the keywords; the rules only where no keyword is found |

- **coverage** = share of businesses that get a section
- **accuracy** = among them, share with the same section as the official main section
- **right overall** = coverage × accuracy (share of all businesses that get the right section)
- **division accuracy** = among businesses that get a division, share with the official main division

The last table checks the leakage worry: the method must be about as accurate for **disappeared** businesses as for
**survived** ones (labelled cohorts only).

In [ ]:
KEYWORDS = pd.concat([name_keywords, description_keywords])
NAME_KW = name_keywords[['section', 'division', 'precision']].to_dict('index')
DESC_KW = description_keywords[['section', 'division']].to_dict('index')

def keyword_code(name):
    """(section, division, word, source) from the best keyword in the name, or empty."""
    found = [(NAME_KW[w]['precision'], ' ' in w, w) for w in words_of(name) if w in NAME_KW]
    if found:
        _, _, word = max(found)
        return NAME_KW[word]['section'], NAME_KW[word]['division'], word, 'names'
    for word in words_of(name):
        if word in DESC_KW:
            return DESC_KW[word]['section'], DESC_KW[word]['division'], word, 'description'
    return '', '', '', ''

def with_keywords(table, column='name'):
    codes = pd.DataFrame([keyword_code(n) for n in table[column]], index=table.index,
                         columns=['kw_section', 'kw_division', 'kw_word', 'kw_source'])
    return table.join(codes)

def methods(table, rules='latest'):
    """rules='latest': the rules' code of the latest name (testing); 'start': of the start name (for the model)."""
    p = 'ssic_latest_' if rules == 'latest' else 'ssic_'
    sure = table[p + 'flag'].isin(['ok', 'division_unclear'])
    rules_s = table[p + 'section'].where(sure, '')
    rules_d = table[p + 'division'].where(sure, '').str.zfill(2).where(lambda s: s != '00', '')
    kw_s, kw_d = table['kw_section'], table['kw_division']
    return {'rules': (rules_s, rules_d),
            'keywords': (kw_s, kw_d),
            'rules, then keywords': (rules_s.where(rules_s != '', kw_s), rules_d.where(rules_s != '', kw_d)),
            'keywords, then rules': (kw_s.where(kw_s != '', rules_s), kw_d.where(kw_s != '', rules_d))}

def score(table):
    out = {}
    for method, (section, division) in methods(table).items():
        has, has_div = section != '', division != ''
        right = section[has] == table.loc[has, 'main_section']
        out[method] = {'coverage': has.mean(), 'accuracy': right.mean(), 'right overall': right.sum() / len(table),
                       'division accuracy': (division[has_div] == table.loc[has_div, 'main_division']).mean()}
    return pd.DataFrame(out).T

test = with_keywords(test)
percent = lambda t: t.to_string(formatters={c: '{:.1%}'.format for c in t.columns})
print(f'Test set: {len(test):,} businesses')
print(percent(score(test)))
print()
from_names = test['kw_source'] == 'names'
from_desc = test['kw_source'] == 'description'
print(f'Keyword accuracy by source: from names {(test.loc[from_names, "kw_section"] == test.loc[from_names, "main_section"]).mean():.1%} '
      f'({from_names.mean():.1%} of businesses) | from SSIC names '
      f'{(test.loc[from_desc, "kw_section"] == test.loc[from_desc, "main_section"]).mean():.1%} ({from_desc.mean():.1%})')
print()
test_labelled = test[test['cohort_quality'] != 'cannot_judge']
print('Labelled cohorts in the test set, "keywords, then rules", by outcome (the leakage check):')
outcome = test_labelled['survived_1y'].map(OUTCOME)
print(percent(pd.DataFrame({o: score(test_labelled[outcome == o]).loc['keywords, then rules']
                            for o in ['survived', 'disappeared']}).T))

### Cell 13 — Current rules that most often disagree with the official sector
For each rule of `isic_classifier.py` (`ssic_label`, flag ok), the share of its businesses whose official main
section is the same. Rules with low agreement are the first to fix (or to let the keywords decide).

In [ ]:
rules_ok = known[known['ssic_latest_flag'] == 'ok']
by_rule = (rules_ok.assign(same=rules_ok['ssic_latest_section'] == rules_ok['main_section'])
           .groupby(['ssic_latest_label', 'ssic_latest_section'])
           .agg(businesses=('same', 'size'), agreement=('same', 'mean')))
most_common_official = (rules_ok.groupby('ssic_latest_label')['main_section']
                        .agg(lambda s: s.value_counts().index[0]))
by_rule['official section most often'] = most_common_official.reindex(by_rule.index.get_level_values(0)).to_numpy()
worst = by_rule[by_rule['businesses'] >= 200].sort_values('agreement').head(20)
with pd.option_context('display.width', 250, 'display.max_colwidth', 50):
    print(worst.to_string(formatters={'agreement': '{:.1%}'.format}))

### Cell 14 — Keywords for every business, and save
Applies the keywords to the **start name** (the name on day one) of **every** business (found in the activity file or not, survived or not), so
every business gets its sector the same way. Saves, in the data folder (never in Git):
- `ssic_keywords.csv`: the keyword list (word, section, division, how many businesses, precision, source)
- `ssic_keywords_by_business.csv`: for each business, the keyword code and the code of each combined method

The team then chooses the method (Cell 12) and the keywords move into `isic_classifier.py`.

In [ ]:
data = with_keywords(data, column='start_name')     # the name on day one (for the model)
for method, (section, division) in methods(data, rules='start').items():
    slug = method.replace(', then ', '_then_')
    data[f'{slug}_section'], data[f'{slug}_division'] = section, division

coverage = pd.Series({m: (s != '').mean() for m, (s, _) in methods(data, rules='start').items()})
labelled_rows = data['cohort_quality'] != 'cannot_judge'
coverage_labelled = pd.Series({m: (s[labelled_rows] != '').mean() for m, (s, _) in methods(data, rules='start').items()})
print('Businesses that get a section:')
print(pd.DataFrame({'all businesses': coverage, 'labelled cohorts': coverage_labelled}).to_string(formatters={
    'all businesses': '{:.1%}'.format, 'labelled cohorts': '{:.1%}'.format}))

KEYWORDS.to_csv(DATA_DIR / 'ssic_keywords.csv', index_label='word', encoding='utf-8-sig')
keep = ['entity_id', 'kw_section', 'kw_division', 'kw_word', 'kw_source'] + [c for c in data.columns if c.endswith(('_section', '_division')) and c.startswith(('rules', 'keywords'))]
data[keep].to_csv(DATA_DIR / 'ssic_keywords_by_business.csv', index=False, encoding='utf-8-sig')
print()
print(f'Saved {len(KEYWORDS):,} keywords to {DATA_DIR / "ssic_keywords.csv"}')
print(f'Saved {len(data):,} businesses to {DATA_DIR / "ssic_keywords_by_business.csv"}')

### Notes — results and decision
**Part 1 (official file)**
- File date: end of 2025 (cohort 2025 Q4: 99% found; 2026: 0%)
- Coverage of labelled businesses: 92.4%
- Leakage gap (survived − disappeared): +68.7 points → **not usable directly as a feature**
- Main-activity rule: `main_*` (main = first for 94% of businesses)
- Name rules vs official: section 76.9% (flag ok), division 63%

**Part 2 (keywords)**
- Keywords from names: 3,484 learned (390 single words, 3,094 phrases). Reviewed by hand in
  `ssic_keywords_decisions_reviewed.xlsx`: 1,028 marked Drop (84 single words, 944 phrases: family and person
  names), removed in Cell 10c (`USE_DECISIONS = True`). **Keywords in use: 2,456.**
- Keywords from SSIC names: switched off (`USE_SSIC_NAME_KEYWORDS = False`, only 22.1% accurate)
- Test set (296,321 businesses, latest name), right overall: rules 39.9% | keywords 36.7% |
  rules, then keywords 44.1% | **keywords, then rules 44.8%** (coverage 55.9%, accuracy 80.2%, division accuracy 72.1%)
- Survived vs disappeared accuracy (must be close): 77.1% vs 81.1% → OK. The coverage gap (43.2% vs 23.6%)
  comes from the names themselves (known on day one), not from the keywords
- **Why the names were dropped:** with them, right overall was 47.2% (+2.4 points), but part of that gain comes
  from the same owner being in both the learning and the test part (e.g. one name in 302 businesses), so it would
  not hold for new owners, and "a family name means a sector" cannot be defended. Only activity words are used.
- Method chosen: **keywords, then rules**, applied to the **start name**. Labelled cohorts with a section:
  41.6% (rules only) → **45.7%**. Businesses with no sector are left empty: the model must treat them as
  `"unclassified"` (its own category), not drop them.
- Where it lives: it **stays in this notebook** (not moved into `isic_classifier.py`). The model reads
  `keywords_then_rules_section` / `keywords_then_rules_division` from `ssic_keywords_by_business.csv`, so this
  notebook must run after `01e_ssic_classification` (Cell 19), with the decisions file in the data folder